### Imbalance Handling

Goal: To test different imbalance handling techniques, to improve model recall

Techniques tested:
1. class_weight='balanced' - increases the importance of the minority class during training
2. SMOTE - generates more fraud samples to balance the training data (based on real frauds)
3. Undersampling - reduces the number of majority class samples to create a more balanced dataset

### Comparison

All techniques tested on Logistic Regression with 5-fold StratifiedKFold CV.
Main metric: PR-AUC. Recall and Precision tracked alongside.
Compared against baseline LR from notebook 02.

Random Forest performed better in the baseline notebook (02), but we test imbalance handling techniques on Logistic Regression. The goal here is to isolate the effect of each technique, so we keep the model fixed and only change the technique. LR is faster and more sensitive to imbalance, which makes the differences between techniques easier to see. 

After selecting the best technique, we'll apply it to stronger models (XGBoost, LightGBM).

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [ ]:
data = pd.read_csv('../data/creditcard.csv')

X = data.drop(columns=["Class"])
y = data["Class"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('scaler', StandardScaler(), ['Amount', 'Time'])
    ],
    remainder='passthrough'
)

pipeline_class_weight = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
])

pipeline_smote = Pipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

pipeline_undersampling = Pipeline([
    ('preprocessor', preprocessor),
    ('undersampler', RandomUnderSampler(random_state=42)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

In [ ]:
scoring = ['average_precision', 'precision', 'recall', 'f1', 'roc_auc']
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_class_weight = cross_validate(pipeline_class_weight, X_train, y_train, cv=skf, scoring=scoring, n_jobs=4)
cv_smote = cross_validate(pipeline_smote, X_train, y_train, cv=skf, scoring=scoring, n_jobs=4)
cv_undersampling = cross_validate(pipeline_undersampling, X_train, y_train, cv=skf, scoring=scoring, n_jobs=4)

In [ ]:
print("LR keys:", cv_class_weight.keys())

In [ ]:
for metric in ['test_average_precision', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc']:
    print(f"class weight {metric} mean: {cv_class_weight[metric].mean():.3f}, std: {cv_class_weight[metric].std():.3f}")

print()

for metric in ['test_average_precision', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc']:
    print(f"SMOTE {metric} mean: {cv_smote[metric].mean():.3f}, std: {cv_smote[metric].std():.3f}")

print()

for metric in ['test_average_precision', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc']:
    print(f"undersampling {metric} mean: {cv_undersampling[metric].mean():.3f}, std: {cv_undersampling[metric].std():.3f}")

### Results

| Technique | PR-AUC | Recall | Precision | F1 |
|---|---|---|---|---|
| Baseline (no handling) | 0.762 | 0.642 | 0.875 | 0.740 |
| class_weight='balanced' | 0.757 | 0.914 | 0.063 | 0.118 |
| SMOTE | 0.753 | 0.916 | 0.058 | 0.109 |
| Undersampling | 0.676 | 0.924 | 0.037 | 0.071 |

### Key observations
1. All three techniques increased recall from ~64% to over 90%.
2. The improvement in recall came with a huge drop in precision (~88% → ~6%).
3. PR-AUC stayed almost the same as the baseline (~0.76).
4. Undersampling performed the worst, likely because it removes a lot of data.
5. SMOTE and class weighting gave similar results.

### Selected technique

Selected: class_weight='balanced'

It performs almost the same as SMOTE on PR-AUC, but is simpler (just one parameter on the model, no extra step in the pipeline) and faster (no additional data generation)

### XGBoost and LightGBM with class balancing

Now we apply the selected technique to stronger models. XGBoost and LightGBM use `scale_pos_weight` parameter (equivalent to `class_weight='balanced'`).

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

pipeline_XGB = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42))
])

pipeline_LGBM = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1))
])

In [ ]:
cv_XGB = cross_validate(pipeline_XGB, X_train, y_train, cv=skf, scoring=scoring, n_jobs=4)
cv_LGBM = cross_validate(pipeline_LGBM, X_train, y_train, cv=skf, scoring=scoring, n_jobs=4)

In [ ]:
print("=== XGB (scale_pos_weight) ===")
for metric in ['test_average_precision', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc']:
    name = metric.replace('test_', '')
    print(f"{name:<18}: {cv_XGB[metric].mean():.3f} ± {cv_XGB[metric].std():.3f}")

print()

print("=== LGBM (class_weight='balanced') ===")
for metric in ['test_average_precision', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc']:
    name = metric.replace('test_', '')
    print(f"{name:<18}: {cv_LGBM[metric].mean():.3f} ± {cv_LGBM[metric].std():.3f}")

### Final models comparison

Each model uses the imbalance technique typical for it:
- LR: class_weight='balanced' (winner from previous comparison)
- RF: no handling (baseline from notebook 02)
- XGB: scale_pos_weight (equivalent of class_weight for XGB)
- LGBM: class_weight='balanced' (since scale_pos_weight breaks LGBM)

| Technique | PR-AUC | Recall | Precision | F1 |
|---|---|---|---|---|
| LR class_weight='balanced' | 0.757 | 0.914 | 0.063 | 0.118 |
| RF baseline (from 02) | 0.840 | 0.766 | 0.940 | 0.844 |
| XGB (scale_pos_weight) | 0.855 | 0.817 | 0.907 | 0.858 |
| LGBM (class_weight='balanced') | 0.830 | 0.807 | 0.843 | 0.822 |

### Key Observations
- Best model: XGB - highest PR-AUC and F1 (although Recall is a bit higher for LGBM)
- Logistic regression model has very low F1 (0.118), because it marks a lot of normal transactions as Frauds
- **Tree-based models dominate** over linear
